# Resnet-18 for KP and AB 

In [ ]:
%pip install h5py mat73

In [ ]:
import numpy as np

KP_STRAINS = [7, 12, 13, 14, 15]
AB_STRAINS = [17, 18, 19, 20]   # 4 strains now
ORDER = KP_STRAINS + AB_STRAINS  # total_strains = 9

def _load_mat_any(path):
    try:
        import mat73
        return mat73.loadmat(path)
    except Exception:
        import scipy.io
        return scipy.io.loadmat(path, squeeze_me=True, struct_as_record=False, simplify_cells=True)

def f_read_bacteria_AC_AB_py(mat_path: str, n_cells_perS: int):
    """
    Python replica with dynamic number of strains.

    Returns:
      X              -> float32 [60, 60, total_strains*n_cells_perS]
      strain_ids_0   -> int32   [total_strains]  (original strain numbers)
      strain_ids     -> int32   [total_strains]  (1..total_strains)
      strain_id      -> int32   [N, 1]  (values 1..total_strains)
      class_id       -> int32   [N, 1]  (0 for KP, 1 for AB)
    """
    data = _load_mat_any(mat_path)

    blocks, strain_id_list, class_id_list = [], [], []
    for j, strain in enumerate(ORDER, start=1):
        key = f'class_{strain}'
        if key not in data:
            raise KeyError(f"Expected key '{key}' not found in '{mat_path}'")
        cells = data[key]
        cells = list(cells.ravel()) if hasattr(cells, 'ravel') else list(cells)
        if len(cells) < n_cells_perS:
            raise ValueError(f"{key} has only {len(cells)} items, need {n_cells_perS}")

        imgs = []
        for i in range(n_cells_perS):
            img = np.asarray(cells[i]['image'], dtype=np.float32)  # 60x60
            imgs.append(img)
            strain_id_list.append(j)
            class_id_list.append(0 if strain in KP_STRAINS else 1)

        blocks.append(np.stack(imgs, axis=2))

    X = np.concatenate(blocks, axis=2).astype(np.float32)
    total_strains = len(ORDER)
    strain_ids_0 = np.asarray(ORDER, dtype=np.int32)
    strain_ids   = np.arange(1, total_strains + 1, dtype=np.int32)
    strain_id    = np.asarray(strain_id_list, dtype=np.int32).reshape(-1, 1)
    class_id     = np.asarray(class_id_list, dtype=np.int32).reshape(-1, 1)
    return X, strain_ids_0, strain_ids, strain_id, class_id


In [ ]:
# Test set version:
X_test, s0_t, s_t, sid_t, cid_t = f_read_bacteria_AC_AB_py(
    '/home/dineth/bacteria_QPM/data/bacteria_test_full_data.mat',
    n_cells_perS=2000
)

In [ ]:
print("X_test shape:", X_test.shape)
print("strain_ids_0:", s0_t)       # Original strain numbers
print("strain_ids:", s_t)          # Mapped 1–10 indices
print("strain_id shape:", sid_t.shape)
print("class_id shape:", cid_t.shape)
print("Unique strain IDs:", np.unique(sid_t))
print("Unique class IDs:", np.unique(cid_t))

In [ ]:
import pandas as pd

counts = pd.DataFrame({
    "strain_id": sid_t.flatten(),
    "class_id": cid_t.flatten()
}).value_counts().reset_index(name="count")

print(counts)

In [ ]:
for c in np.unique(cid_t):
    strain_subset = np.unique(sid_t[cid_t.flatten() == c])
    print(f"Class {c}: strains {strain_subset}")

In [ ]:
# # Train set 
# X, strain_ids_0, strain_ids, strain_id, class_id = f_read_bacteria_AC_AB_py(
#     '/home/dineth/bacteria_QPM/data/bacteria_train_full_data.mat',
#     n_cells_perS=16000
# )

In [ ]:
import numpy as np
import scipy.io
from scipy.io.matlab.mio5_params import mat_struct

KP = [7, 12, 13, 14, 15]
AB = [17, 18, 19, 20]  # no 16, no 0
TARGET_ORDER = KP + AB

# converters and finders
def _deep(o):
    if isinstance(o, mat_struct):
        return {f: _deep(getattr(o, f)) for f in o._fieldnames}
    if isinstance(o, np.ndarray):
        if o.dtype == object:
            if o.ndim == 0: return _deep(o.item())
            return [_deep(x) for x in o.ravel()]
        if o.dtype.names is not None:
            if o.shape == ():  # scalar structured
                return {n: _deep(o[n]) for n in o.dtype.names}
            return [{n: _deep(o[idx][n]) for n in o.dtype.names}
                    for idx in np.ndindex(o.shape)]
        if o.ndim == 0: return o.item()
        return o
    if isinstance(o, np.void) and o.dtype.names is not None:
        return {n: _deep(o[n]) for n in o.dtype.names}
    return o

def _load_var_v5(mat_path, varname):
    part = scipy.io.loadmat(
        mat_path,
        variable_names=[varname],
        squeeze_me=False,
        struct_as_record=True,
        chars_as_strings=True,
    )
    if varname not in part:
        raise KeyError(f"{varname} not found in {mat_path}")
    return _deep(part[varname])

def _coerce_60x60(x):
    a = np.asarray(x, dtype=np.float32)
    a = np.squeeze(a)
    if a.shape == (60, 60):
        return a
    if a.ndim == 3:
        for ax in range(3):
            if a.shape[ax] == 1:
                b = np.squeeze(a, axis=ax)
                if b.shape == (60, 60):
                    return b.astype(np.float32)
    raise ValueError(f"Unexpected image shape {a.shape}")

def _find_image(o):
    if isinstance(o, dict):
        if "image" in o: return _coerce_60x60(o["image"])
        for v in o.values():
            try: return _find_image(v)
            except Exception: pass
        raise KeyError("No 'image' in dict")
    if isinstance(o, (list, tuple)):
        for v in o:
            try: return _find_image(v)
            except Exception: pass
        raise KeyError("No 'image' in list/tuple")
    if isinstance(o, np.ndarray):
        if o.dtype == object:
            for v in o.ravel():
                try: return _find_image(v)
                except Exception: pass
            raise KeyError("No 'image' in object array")
        if o.dtype.names is not None:
            for n in o.dtype.names:
                try: return _find_image(o[n])
                except Exception: pass
            raise KeyError("No 'image' in structured array")
        return _coerce_60x60(o)
    if isinstance(o, np.void) and o.dtype.names is not None:
        for n in o.dtype.names:
            try: return _find_image(o[n])
            except Exception: pass
        raise KeyError("No 'image' in np.void")
    raise TypeError(f"Unsupported item type {type(o)}")


def f_read_bacteria_AC_AB_py(mat_path: str, n_cells_perS: int):
    """
    Loads KP vs AB images robustly, probing expected strains directly.
    Uses only the first n_cells_perS items per successfully loaded strain.
    Returns:
      X -> float32 [60, 60, S*n_cells_perS]
      strain_ids_0 -> int32 [S] original strain numbers in order used
      strain_ids   -> int32 [S] 1..S
      strain_id    -> int32 [S*n_cells_perS, 1]
      class_id     -> int32 [S*n_cells_perS, 1]  (0=KP, 1=AB)
    """
    usable_order = []
    loaded_cells_per_strain = {}

    # Try to load each target strain individually
    for s in TARGET_ORDER:
        name = f'class_{s}'
        try:
            cells = _load_var_v5(mat_path, name)
            # normalize to list
            if isinstance(cells, np.ndarray) and cells.dtype == object:
                cells = [_deep(x) for x in cells.ravel()]
            elif not isinstance(cells, list):
                cells = [cells]
            if len(cells) >= n_cells_perS:
                usable_order.append(s)
                loaded_cells_per_strain[s] = cells
            else:
                raise ValueError(f"{name} has only {len(cells)} items (< {n_cells_perS})")
        except Exception as e:
            print(f"✖ Skipping {name}: {e}")
            continue

    if not usable_order:
        raise RuntimeError("No expected class_* variables could be loaded from this file.")

    S = len(usable_order)
    H = W = 60
    N = S * n_cells_perS
    X = np.empty((H, W, N), dtype=np.float32)
    strain_id = np.empty((N, 1), dtype=np.int32)
    class_id  = np.empty((N, 1), dtype=np.int32)

    ptr = 0
    for j, strain in enumerate(usable_order, start=1):
        cells = loaded_cells_per_strain[strain]
        # take first n_cells_perS
        for i in range(n_cells_perS):
            X[:, :, ptr + i] = _find_image(cells[i])
        strain_id[ptr:ptr+n_cells_perS, 0] = j
        class_id[ptr:ptr+n_cells_perS, 0]  = (0 if strain in KP else 1)
        ptr += n_cells_perS

    strain_ids_0 = np.asarray(usable_order, dtype=np.int32)
    strain_ids   = np.arange(1, S + 1, dtype=np.int32)
    return X, strain_ids_0, strain_ids, strain_id, class_id

In [ ]:
# TRAIN
X_tr, s0_tr, s_tr, sid_tr, cid_tr = f_read_bacteria_AC_AB_py(
    '/home/dineth/bacteria_QPM/data/bacteria_train_full_data.mat',
    n_cells_perS=16000
)

# VAL
X_va, s0_va, s_va, sid_va, cid_va = f_read_bacteria_AC_AB_py(
    '/home/dineth/bacteria_QPM/data/bacteria_val_full_data.mat',
    n_cells_perS=2000
)

# TEST
X_te, s0_te, s_te, sid_te, cid_te = f_read_bacteria_AC_AB_py(
    '/home/dineth/bacteria_QPM/data/bacteria_test_full_data.mat',
    n_cells_perS=2000
)


In [ ]:
print("\n TRAIN")
print("X_tr shape:", X_tr.shape)
print("strain_ids_0:", s0_tr)
print("strain_ids:", s_tr)
print("strain_id shape:", sid_tr.shape, "| unique:", np.unique(sid_tr))
print("class_id shape:", cid_tr.shape, "| unique:", np.unique(cid_tr))
print("Counts per class:")
for c in np.unique(cid_tr):
    count = np.sum(cid_tr == c)
    print(f"  Class {c}: {count} samples")

print("\n VAL ")
print("X_va shape:", X_va.shape)
print("strain_ids_0:", s0_va)
print("strain_ids:", s_va)
print("strain_id shape:", sid_va.shape, "| unique:", np.unique(sid_va))
print("class_id shape:", cid_va.shape, "| unique:", np.unique(cid_va))

print("\n TEST ")
print("X_te shape:", X_te.shape)
print("strain_ids_0:", s0_te)
print("strain_ids:", s_te)
print("strain_id shape:", sid_te.shape, "| unique:", np.unique(sid_te))
print("class_id shape:", cid_te.shape, "| unique:", np.unique(cid_te))


# Now the permutation test

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import random

# Set random seeds for reproducibility
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")


In [ ]:
class ResNet18BinaryClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet18BinaryClassifier, self).__init__()
        # Load pre-trained ResNet-18
        self.resnet = models.resnet18(pretrained=False)  # No pretrained weights for 60x60 images
        
        # Modify the first layer to accept single channel (grayscale) 60x60 images
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Modify the final layer for binary classification
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)

In [ ]:
def prepare_data(X, y, batch_size=32, shuffle=True):
    """Convert numpy arrays to PyTorch DataLoader"""
    # Reshape X from (60, 60, N) to (N, 1, 60, 60) for PyTorch
    X_reshaped = np.transpose(X, (2, 0, 1))  # (N, 60, 60)
    X_reshaped = np.expand_dims(X_reshaped, axis=1)  # (N, 1, 60, 60)
    
    X_tensor = torch.FloatTensor(X_reshaped)
    y_tensor = torch.LongTensor(y.flatten())
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # Calculate accuracy
        bs = target.size(0)
        total_loss += loss.detach().item() * bs
        pred = output.detach().argmax(dim=1)
        correct += (pred == target).sum().item()
        total += bs

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc

In [ ]:
def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            bs = target.size(0)
            total_loss += loss.detach().item() * bs
            pred = output.detach().argmax(dim=1)

            correct += (pred == target).sum().item()
            total += bs
            all_preds.extend(pred.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc, all_preds, all_targets

In [ ]:
def train_eval_permutation_with_epochs(
    X_train, y_train, X_test, y_test, epochs=10, lr=0.01, batch_size=64
):
    """Train ResNet-18 with SGD optimizer for permutation test"""
    
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    
    # Create new model for each permutation
    model = ResNet18BinaryClassifier().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    train_loader = prepare_data(X_train, y_train, batch_size=batch_size, shuffle=True)
    test_loader = prepare_data(X_test, y_test, batch_size=batch_size, shuffle=False)

    best_test_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        
        # Evaluate on test set
        model.eval()
        with torch.no_grad():
            test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)

        print(f"    Epoch {epoch}: Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"[BEST] Epoch {best_epoch}: Test Acc = {best_test_acc:.2f}%")
    return best_test_acc / 100.0


In [ ]:
# Define strain mappings
KP_STRAINS = [7, 12, 13, 14, 15]
AB_STRAINS = [17, 18, 19, 20]
ORDER = KP_STRAINS + AB_STRAINS

# Binary class mapping (KP=0, AB=1)
CLASS_TO_BIN = {0: 0, 1: 1}  # KP -> 0, AB -> 1

LAST_PERM_MAP = None

def _make_strain_perm_map(focused_strains, rng):
    n_kp = sum(1 for s in focused_strains if s in KP_STRAINS)  # KP strains
    n_ab = sum(1 for s in focused_strains if s in AB_STRAINS)  # AB strains

    # Build a label list of the correct total length, then shuffle
    perm_bins = np.array([0] * n_kp + [1] * n_ab, dtype=int)
    rng.shuffle(perm_bins)

    # Safety check
    assert len(perm_bins) == len(focused_strains), \
        f"perm_bins({len(perm_bins)}) != focused_strains({len(focused_strains)})"

    # Assign one permuted binary label per strain
    return {s: perm_bins[i] for i, s in enumerate(focused_strains)}

def print_permutation_mapping(perm_idx, y_ref_perm, y_te_perm):
    print(f"\nPermutation {perm_idx}: strain -> class (TRUE -> PERMUTED)")
    if LAST_PERM_MAP is None:
        print("  [warn] LAST_PERM_MAP is not set")
        return
    for sid in ORDER:
        true_class = 0 if sid in KP_STRAINS else 1
        perm_class = LAST_PERM_MAP[sid]
        print(f"Strain {sid:2d} | {'KP' if true_class == 0 else 'AB':<2} -> {'KP' if perm_class == 0 else 'AB'}")

# Permutation test parameters
N_PERMUTATIONS = 1000  # Number of permutations
PERM_EPOCHS = 10       # Epochs for each permutation
RANDOM_SEED = 42

In [ ]:
# Cell 19: True Labels Training
print(f"\nTRUE labels training:")
true_accuracy = train_eval_permutation_with_epochs(
    X_tr, cid_tr,  # Using train data
    X_te, cid_te,  # Using test data
    epochs=PERM_EPOCHS
)
print(f"TRUE labels final accuracy: {true_accuracy:.2%}")

In [ ]:
import itertools
import math

# Calculate the exact number of unique permutations
total_strains = 9
kp_strains = 5  # number of 0s
ab_strains = 4  # number of 1s

unique_permutations_count = math.factorial(total_strains) // (math.factorial(kp_strains) * math.factorial(ab_strains))
print(f"Number of unique permutations: {unique_permutations_count}")

# Generate all unique permutations
def generate_all_unique_permutations():

    # Create the base pattern: 5 zeros and 4 ones
    base_pattern = [0] * kp_strains + [1] * ab_strains
    
    # Generate all unique permutations of this pattern
    unique_patterns = list(set(itertools.permutations(base_pattern)))
    
    print(f"Generated {len(unique_patterns)} unique patterns")
    return unique_patterns

# Generate all unique permutation patterns
all_unique_patterns = generate_all_unique_permutations()

# Updated permutation test to use all unique permutations
N_PERMUTATIONS = len(all_unique_patterns)  # This will be 126
PERM_EPOCHS = 10
RANDOM_SEED = 42

print(f"\nRunning {N_PERMUTATIONS} unique permutations...")

# Get strain IDs for each sample - use original strain numbers, not mapped IDs
# Create mapping from mapped strain IDs (1-9) back to original strain numbers
id_to_original = {mapped_id: original_id for mapped_id, original_id in zip(s_tr, s0_tr)}

# Get strain IDs for each sample - convert mapped IDs back to original strain numbers
ref_strains_order = np.array([id_to_original[sid] for sid in sid_tr.flatten()])
te_strains_order = np.array([id_to_original[sid] for sid in sid_te.flatten()])

null_accuracies = []

for perm_idx, perm_pattern in enumerate(all_unique_patterns, 1):
    # Create strain-to-label mapping using the current unique pattern
    strain_to_label = {strain: perm_pattern[i] for i, strain in enumerate(ORDER)}
    
    # Apply the mapping to both splits
    y_ref_permuted = np.array([strain_to_label[s] for s in ref_strains_order], dtype=int)
    y_te_permuted = np.array([strain_to_label[s] for s in te_strains_order], dtype=int)
    
    LAST_PERM_MAP = strain_to_label
    print_permutation_mapping(perm_idx, y_ref_permuted, y_te_permuted)
    
    # Train & evaluate with permuted labels
    print(f"\nPermutation {perm_idx} training:")
    perm_accuracy = train_eval_permutation_with_epochs(
        X_tr, y_ref_permuted,
        X_te, y_te_permuted,
        epochs=PERM_EPOCHS
    )
    null_accuracies.append(perm_accuracy)
    
    if perm_idx % 20 == 0:  # Progress update every 20 permutations
        print(f"Completed {perm_idx}/{N_PERMUTATIONS} permutations")

print("Permutation test completed!")

In [ ]:
# Create histogram visualization for permutation test
null_accuracies = np.array(null_accuracies)

plt.figure(figsize=(10, 6))
plt.hist(null_accuracies * 100, bins=30, color='navy', alpha=0.7, edgecolor='black')

# Observed performance line
plt.axvline(true_accuracy * 100, color='red', linewidth=3, label='True Performance')

# Chance baseline for binary classification = 50%
plt.axvline(50, color='k', linestyle=':', linewidth=2, label='Chance (50%)')

plt.xlabel("Accuracy [%] (on the test set)")
plt.ylabel("Frequency")
plt.title("Permutation Test Results: KP vs AB Classification\n(ResNet-18 with SGD)")

# Add text annotation with observed accuracy and permutation p-value
ymax = max(np.histogram(null_accuracies * 100, bins=30)[0])
p_value = np.mean(null_accuracies >= true_accuracy)

plt.text(true_accuracy * 100 + 2, ymax * 0.7,
         f"Observed = {true_accuracy * 100:.2f}%\nP-value = {p_value:.4f}",
         fontsize=12, bbox=dict(facecolor='yellow', alpha=0.7))

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Cell 22: Statistical Summary
print("\n" + "="*60)
print("PERMUTATION TEST RESULTS SUMMARY")
print("="*60)
print(f"True accuracy: {true_accuracy:.4f} ({true_accuracy*100:.2f}%)")
print(f"Null distribution mean: {np.mean(null_accuracies):.4f} ({np.mean(null_accuracies)*100:.2f}%)")
print(f"Null distribution std: {np.std(null_accuracies):.4f} ({np.std(null_accuracies)*100:.2f}%)")
print(f"P-value: {p_value:.4f}")
print(f"Number of permutations: {N_PERMUTATIONS}")

if p_value < 0.05:
    print("✓ Result is statistically significant (p < 0.05)")
else:
    print("✗ Result is NOT statistically significant (p ≥ 0.05)")

print(f"Percentile of true accuracy in null distribution: {100 * (1 - p_value):.1f}th percentile")

In [ ]:
import math

total_strains = 10
kp_strains = 5  # number of 0s
ab_strains = 5  # number of 1s

unique_permutations = math.factorial(total_strains) // (math.factorial(kp_strains) * math.factorial(ab_strains))

print(f"Total strains: {total_strains}")
print(f"KP strains (class 0): {kp_strains}")
print(f"AB strains (class 1): {ab_strains}")
print(f"Number of unique permutations: {unique_permutations}")